##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## 使用 JORA 微調 Gemma 用於檢索增強生成

為基於檢索的任務擴展大型語言模型（LLM），特別是在檢索增強生成（RAG）中，提出了重大的記憶挑戰，特別是當fine-tuning廣泛的prompt序列時。
[Gemma](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開放模型，採用與創建 Gemini 模型相同的研究和技術構建。它們是文字到文字、僅限解碼器的大型語言模型，提供英文版本，具有開放權重、預訓練變體和指令調整變體。 Gemma 模型非常適合各種文本生成任務，包括問答、摘要和推論。它們的尺寸相對較小，因此可以將它們部署在資源有限的環境中，例如筆記型電腦、桌上型電腦或您自己的雲端基礎設施，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
現有的開源 library 支援跨多個 GPU 的完整模型 inference 和 fine-tuning，但通常無法有效分配檢索上下文所需的參數。為了解決這個限制，[JORA](https://github.com/aniquetahir/JORA) 引入了一種新穎的framework，用於使用分佈式訓練對 Llama/Gemma 模型進行參數高效微調 (PEFT)，利用 [JAX](https://jax.readthedocs.io/en/latest/)。此framework 獨特地利用JAX 的即時(JIT) 編譯和張量分片來實現高效的資源管理，從而在減少內存需求的情況下實現加速fine-tuning。這項進步顯著提高了 fine-tuning LLM 對於複雜 RAG 應用程式的可擴展性和可行性，即使在 GPU 資源有限的系統上也是如此。
實驗表明，與使用四個 GPU 的 [Hugging Face](https://huggingface.co/docs/transformers/en/main_classes/trainer)/[DeepSpeed](https://github.com/microsoft/DeepSpeed) 實作相比，runtime** 效能提升了 **12 倍以上，而每個 GPU 消耗的 VRAM 不到一半。
在本教學中，您將了解使用 JORA fine-tuning [Gemma](https://github.com/google/gemma) 模型的端到端過程，並將訓練後的模型轉換回 inference 的 [Hugging Face](@@P0001@) 格式。

<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Finetune_with_JORA.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>
<br><br>
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)]("https://www.kaggle.com/notebooks/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_2]Finetune_with_JORA.ipynb")

## 設定


### 選擇執行時環境

首先，您可以選擇 **Google Colab** 或 **Kaggle** 作為您的平台。選擇一個，然後從那裡繼續。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>

  1. 按一下「**在 Colab** 中開啟」。
  2. 您需要存取具有足夠資源的 [**Colab Pro/Pro+**](https://colab.research.google.com/signup) runtime 來執行 Gemma 模型。
  3. In the menu, go to **Runtime** > **Change runtime type**.
  4. 確保 **GPU** 設定為 **A100**。

- #### **Kaggle** <img src="https://upload.wikimedia.org/wikipedia/commons/7/7c/Kaggle_logo.png" alt="Kaggle" width="40"/>

  1. 按一下「**在 Kaggle** 中開啟」。
  2. 點選右側邊欄中的**會話選項**。
  3. 在 **加速器** 下，選擇 **GPU T4 x2**。
- 注意：此執行個體配備 **15 GB x2**（每個 T4 GPU 15 GB）VRAM 和 **30 GB** RAM。  4. 儲存設置，notebook 將在 GPU 支援下重新啟動。

### Gemma設置

#### **Kaggle 型號**

要完成本教學並使用必要的 Kaggle Gemma Flax 模型下載和微調，您首先需要完成 [Gemma 設定](https://ai.google.dev/gemma/docs/setup) 中的設定說明。 Gemma 設定說明向您展示如何執行以下操作：
* 在 kaggle.com 上造訪Gemma。
* 選擇具有足夠資源執行的 Colab/Kaggle runtime
Gemma 模型。* 您將在本指南後面產生並設定 Kaggle 使用者名稱和 API 金鑰作為 Colab secrets。

#### **Hugging Face Hub**

您還需要登入Hugging Face Hub 下載fine-tuning 時使用的確切Gemma 模型，以便您可以將Flax 模型轉換為Hugging Face 格式並稍後執行inference。讓我們為您設定Gemma：
1. **Hugging Face 帳戶：** 如果您還沒有帳戶，您可以點選[此處](https://huggingface.co/join) 建立免費的Hugging Face 帳戶。
2. **Gemma 模型存取：** 前往 [Gemma 模型頁面](https://huggingface.co/collections/google/gemma-release-65d5efbccdbb8c4202ec078b) 並接受使用條件。
3. **Colab/Kaggle 和 Gemma 功能：** 對於本教學，您需要 Colab/Kaggle runtime 具有足夠的資源來處理 Gemma 模型。啟動Colab/Kaggle 會話時選擇適當的runtime。
4. **Hugging Face token：** 透過點選[此處](https://huggingface.co/settings/tokens) 產生Hugging Face 存取權限（最好是`write` 權限）token。這個token稍後會派上用場。

完成 Gemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。

### 設定您的憑證

要存取私有模型和dataset，您需要登入Hugging Face（HF）和Kaggle生態系統。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>
如果您使用 Colab，您可以使用 Colab Secrets manager 安全地儲存 Hugging Face token (`HF_TOKEN`)：  1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
  2. **新增Hugging Facetoken**：
- 建立一個新的secret，其**名稱**為`HF_TOKEN`。 - 將token 金鑰複製/貼上到`HF_TOKEN` 的**值** 輸入框中。 - **切換**左側的按鈕以允許notebook訪問secret  3. **新增Kaggletoken**：
- 與之前相同，但對`KAGGLE_USERNAME` 和`KAGGLE_KEY` 重複此操作。

- #### **Kaggle** <img src="https://upload.wikimedia.org/wikipedia/commons/7/7c/Kaggle_logo.png" alt="Kaggle" width="40"/>
要在此 notebook 中安全地使用 Hugging Face token (`HF_TOKEN`)，您需要將其作為 secret 添加到 Kaggle 環境中：  1. 開啟 Kaggle notebook 並找到 notebook 介面頂部的 **插件** 選單。
  2. 點選 **Secrets** 來管理您的環境secrets。
<img src="https://i.imgur.com/vxrtJuM.png" alt="The Secrets option is found at the top." width=50%>  3. **新增Hugging Facetoken**：
- 點選「**新增secret**」按鈕。 - 在**標籤**欄位中，輸入`HF_TOKEN`。 - 在 **值** 欄位中，貼上 Hugging Face token。 - 點選「**儲存**」新增secret。  4. **新增Kaggletoken**：
- 與之前相同，但對`KAGGLE_USERNAME` 和`KAGGLE_KEY` 重複此操作。

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import userdata
    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
elif os.path.exists('/kaggle/working'):
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret('KAGGLE_KEY')
else:
    raise RuntimeError(
        "Unsupported runtime environment detected.\n"
        "This notebook currently supports execution on Google Colab or Kaggle.\n"
        "Please ensure you are running in one of these environments.\n"
        "If you are running locally or on a different platform, manually set the following environment variables:\n"
        " - HF_TOKEN\n"
        " - KAGGLE_USERNAME\n"
        " - KAGGLE_KEY\n\n"
        "You can set environment variables in your terminal or within your Python notebook before running any cells."
    )

# Disable progress bar to prevent verbose logging by kagglehub
os.environ["TQDM_DISABLE"] = "1"

### 克隆 **JORA** 並安裝依賴項

In [ ]:
# Clone the JORA repository and install the requirements
!git clone https://github.com/aniquetahir/JORA.git
%cd JORA
!pip install -q -e .

# Install google-deepmind/gemma as it's a required dependency for JORA
!pip install -q git+https://github.com/google-deepmind/gemma.git

# Install the appropriate JAX version
JAX_VERSION = "0.4.33"
!pip install -U --pre -f https://storage.googleapis.com/jax-releases/jax_nightly_releases.html \
  jax==$JAX_VERSION jaxlib==$JAX_VERSION \
  jax-cuda12-plugin[with_cuda]==$JAX_VERSION jax-cuda12-pjrt==$JAX_VERSION

Cloning into 'JORA'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (299/299), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 299 (delta 151), reused 203 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (299/299), 6.99 MiB | 17.66 MiB/s, done.
Resolving deltas: 100% (151/151), done.
/content/JORA
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.1/320.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 6.3 MB/s eta 0:00:00
   ━━━

### 導入相依性

In [ ]:
# Patch JORA's initialisation.py file to be compatible with the latest JAX version

!sed -i "s/jax\.config\.update('jax_default_matmul_precision', *jax\.lax\.Precision\.HIGHEST)/jax.config.update('jax_default_matmul_precision', 'bfloat16')/" jora/lib/proc_init_utils/initialisation.py

In [ ]:
import kagglehub
import jax
import jora
import pathlib
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

## 下載Gemma模型

現在，您可以使用`kagglehub`下載Gemma模型：

In [ ]:
VARIANT = "gemma2-2b-it"
GEMMA_PATH = kagglehub.model_download(f'google/gemma-2/Flax/{VARIANT}')
print('GEMMA_PATH:', GEMMA_PATH)

GEMMA_PATH: /root/.cache/kagglehub/models/google/gemma-2/Flax/gemma2-2b-it/1


In [ ]:
# Note: JORA only supports loading Gemma and Gemma 1.1 models at the moment
# Let's add an entry for `gemma2-2b-it` so that the Gemma 2 model can be
# discoverable by JORA

# Allow JORA to discover the newly downloaded Gemma 2 model
JORA_GEMMA_VERSIONS = jora.lib.gemma.gemma_config.GEMMA_VERSIONS
JORA_GEMMA_VERSIONS = JORA_GEMMA_VERSIONS.add('gemma2-2b-it')
print(jora.lib.gemma.gemma_config.GEMMA_VERSIONS)

JORA_GEMMA_MODEL_MAPPING = jora.lib.gemma.common.model_config_mapping
JORA_GEMMA_MODEL_MAPPING = JORA_GEMMA_MODEL_MAPPING.update({
    'gemma2-2b-it': jora.lib.gemma.gemma_config.GemmaConfig2B
})
print(jora.lib.gemma.common.model_config_mapping)

{'7b-it', '2b-it', 'gemma2-2b-it', '7b', '2b'}
{'2b': GemmaConfig(n_heads=8, n_kv=1), '2b-it': GemmaConfig(n_heads=8, n_kv=1), '7b': GemmaConfig(n_heads=16, n_kv=16), '7b-it': GemmaConfig(n_heads=16, n_kv=16), '1.1-2b-it': GemmaConfig(n_heads=8, n_kv=1), '1.1-7b-it': GemmaConfig(n_heads=16, n_kv=16), 'gemma2-2b-it': GemmaConfig(n_heads=8, n_kv=1)}


**注意：** 預設情況下，`kagglehub` 將模型儲存在`~/.cache/kagglehub` 目錄中。
驗證 JAX 是否辨識 GPU 裝置：

In [ ]:
print(jax.devices())

[CudaDevice(id=0)]


## 設定 JORA 並準備 dataset

在這裡，您將設定 Gemma 模型以及 **LoRA** fine-tuning 的訓練過程。
為了微調Gemma，您將使用 **Alpaca** dataset。確保dataset 文件`alpaca_data_cleaned.json` 位於適當的目錄中。您可以從[此處](https://github.com/tatsu-lab/stanford_alpaca/blob/main/alpaca_data_cleaned.json) 下載它或使用儲存庫中捆綁的版本。出於演示目的，我們使用捆綁的產品。
**鳴謝：** [斯坦福羊駝毛](https://github.com/tatsu-lab/stanford_alpaca/blob/main/alpaca_data.json)
`generate_alpaca_dataset` 函數用於從 Alpaca 格式 JSON 檔案產生 dataset。這有助於指導格式訓練，因為 dataset 處理、token 化和批次是由 library 處理的。或者，火炬`Dataset` 和`DataLoader` 可用於客製化dataset。

In [ ]:
# Configure the model and training parameters
config = jora.ParagemmaConfig(
    # Feel free to tweak these parameters
    N_EPOCHS=1,
    LORA_R=8,
    # Note: The `LORA_DROPOUT` parameter is currently not configurable.
    # https://github.com/aniquetahir/JORA?tab=readme-ov-file#contributing
    LORA_ALPHA=16,
    LR=1e-5,
    BATCH_SIZE=2,
    N_ACCUMULATION_STEPS=8,
    GEMMA_MODEL_PATH=GEMMA_PATH,
    MAX_SEQ_LEN=512,
    MODEL_VERSION=VARIANT
)

# Path to the Alpaca dataset
dataset_path = 'jora/alpaca_data_cleaned.json'

# Generate the dataset with a 20% split for prototyping.
# When running on Kaggle, set split_percentage to 0.005 to use a smaller subset
# for quicker demonstration purposes.
dataset = jora.generate_alpaca_dataset_gemma(
    dataset_path, 'train', config,
    # Change the split percentage to '0.005` if you're on Kaggle
    split_percentage=0.2,
    alpaca_mix=0.3
)

Processing data...


`ParagemmaConfig` 類別用於設定訓練設定，而`generate_alpaca_dataset_gemma` 處理dataset、處理token 化並準備訓練。

In [ ]:
config

ParagemmaConfig(GEMMA_MODEL_PATH='/root/.cache/kagglehub/models/google/gemma-2/Flax/gemma2-2b-it/1', MODEL_VERSION='gemma2-2b-it', NUM_SHARDS=None, LORA_R=8, LORA_ALPHA=16, LORA_DROPOUT=0.05, LR=1e-05, BATCH_SIZE=2, N_ACCUMULATION_STEPS=8, MAX_SEQ_LEN=512, N_EPOCHS=1, SEED=420, CACHE_SIZE=30)

## 使用 **JORA** 微調 Gemma

現在，您可以使用`train_lora_gemma` 函數繼續fine-tuning 模型，該函數使用LoRA（低階適應）啟動fine-tuning 過程。 checkpoint將保存在`checkpoint_path`指定的資料夾中。

In [ ]:
# Path to the trained LoRA weights
checkpoint_path = 'checkpoints'
jora.train_lora_gemma(config, dataset, checkpoint_path)

Successfully loaded and sharded model parameters!


Output()

**注意**：對整個 dataset 進行微調可能非常耗時，並且可能會超出 **Kaggle** 上的可用 GPU 配額，或消耗 **Google Colab** 上的大量計算單元。使用較小的分割有助於管理資源使用並保持在平台施加的限制內。

## 將模型轉換為 **Hugging Face 格式**

fine-tuning之後，您需要將訓練好的模型轉換為Hugging Face格式，以相容Hugging Face生態系統，以便以後可以輕鬆執行inference。
**用法：**
```python
lorize_huggingface(HUGGINGFACE_PATH, JAX_PATH, SAVE_PATH, gemma=True)
```

- **HUGGINGFACE_PATH**：Hugging Face Gemma 模型的路徑（fine-tuning 之前的基本模型）。
- **JAX_PATH**：LoRA 合併參數的路徑（經過訓練的LoRA 權重）。
- **SAVE_PATH**：儲存微調Hugging Face Gemma 模型的路徑。
- **gemma**：指示您正在使用 Gemma 模型的標誌。

首先，指定路徑：

In [ ]:
# Specify the repository
repo_id = "google/gemma-2-2b-it"
local_dir = 'pretrained'

snapshot_download(
    repo_id=repo_id,
    local_dir=local_dir,
    revision="main",
    ignore_patterns=['*.gguf']
)

HUGGINGFACE_PATH = local_dir
JAX_PATH = 'checkpoints/jax_lora_final.pickle'
SAVE_PATH = 'gemma-ft'

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

然後，執行轉換器：

In [ ]:
from jora.hf.__main__ import lorize_huggingface

lorize_huggingface(HUGGINGFACE_PATH, JAX_PATH, SAVE_PATH, gemma=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model loaded
model saved to gemma-ft


- `jora.hf` 模組將JAX 訓練的模型轉換回Hugging Face 格式。
- 它將 LoRA 權重與原始模型參數合併。
- The converted model is saved in the specified `SAVE_PATH`.

## 載入模型並生成文本

最後，您可以使用Hugging Face的Transformerslibrary來載入轉換後的模型。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_PATH)
model = AutoModelForCausalLM.from_pretrained(SAVE_PATH, device_map="auto")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

在這裡，tokenizer 和模型都會首先加載，然後模型會自動移動到適當的裝置。最後，您使用模型產生文本，同時依賴 Alpaca prompt 格式：

In [ ]:
# Define the Alpaca prompt template
alpaca_prompt = """\
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

# Function to generate response
def generate_response(instruction, input_text="", max_new_tokens=384):
    prompt = alpaca_prompt.format(instruction, input_text)
    device = "cuda"
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    outputs = model.generate(inputs, max_new_tokens=max_new_tokens)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(text)

In [ ]:
generate_response(
    instruction="Identify 3 common mistakes in the following sentence. Suggest changes.",
    input_text="She seems to believe that the real key to success is working smart and hard."
)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Identify 3 common mistakes in the following sentence. Suggest changes.

### Input:
She seems to believe that the real key to sucsess is working smart and hard.

### Response:
1. "sucsess" should be "success"
2. "seems to believe" is a weak phrase.
3. "working smart and hard" is a cliché.


In [ ]:
generate_response(
    instruction="Make a prediction about what will happen in the next paragraph.",
    input_text="Mary had been living in the small town for many years and had never seen anything like what was coming.",
)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Make a prediction about what will happen in the next paragraph.

### Input:
Mary had been living in the small town for many years and had never seen anything like what was coming.

### Response:
She will be surprised by the event.


In [ ]:
generate_response(
    instruction="Identify a suitable <verb> in the following sentence.",
    input_text="The cat <verb> in the garden.",
)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Identify a suitable <verb> in the following sentence.

### Input:
The cat <verb> in the garden.

### Response:
played


In [ ]:
generate_response(
    instruction="Explain why the quote is appropriate or not for a yoga class.",
    input_text="Don't quit. Suffer now and live the rest of your life as a champion.",
)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Explain why the quote is appropriate or not for a yoga class.

### Input:
Don't quit. Suffer now and live the rest of your life as a champion.

### Response:
This quote is not appropriate for a yoga class because it promotes a competitive mindset and ignores the importance of self-compassion and acceptance.


## 將模型推送到您的Hugging Face Hub


（可選）Hugging Face 讓您輕鬆地將經過訓練的模型儲存在其中心。

In [ ]:
# Note: The token needs to have "write" permission
#       You can check it here:
#       https://huggingface.co/settings/tokens
# Uncomment and run this if you wish to publish the model to Hugging Face Hub
# model.push_to_hub("my-gemma-finetuned-model")

在本教學中，您學習如何使用 JORA 微調 Gemma 模型並將其轉換為 inference 的 Hugging Face 模型格式。透過利用JAX的 JIT 編譯和張量分片功能，您可以實現高效的資源管理，從而在減少記憶體需求的情況下實現加速fine-tuning。